In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 04 - Registro y selección del mejor modelo
# MAGIC Selecciona el mejor run por ROC-AUC, lo registra en Unity Catalog y asigna el alias PRINCIPAL.

# COMMAND ----------

import mlflow
from mlflow import MlflowClient
import pandas as pd

mlflow.set_registry_uri("databricks-uc")
EXPERIMENT_PATH = "/Shared/control-2"
REGISTERED_MODEL = "workspace.control2.modelo_control2"
ALIAS = "PRINCIPAL"
client = MlflowClient()

# COMMAND ----------

experiment = client.get_experiment_by_name(EXPERIMENT_PATH)
if experiment is None:
    raise Exception(f"No existe el experimento: {EXPERIMENT_PATH}")

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.roc_auc DESC"]
)

if not runs:
    raise Exception("No hay runs finalizados.")

results = pd.DataFrame([{
    "run_name": r.data.params.get("run_name", r.info.run_name),
    "modelo": r.data.params.get("modelo", ""),
    "roc_auc": r.data.metrics.get("roc_auc"),
    "accuracy": r.data.metrics.get("accuracy"),
    "precision": r.data.metrics.get("precision"),
    "recall": r.data.metrics.get("recall"),
    "f1_score": r.data.metrics.get("f1_score"),
    "run_id": r.info.run_id
} for r in runs])

display(results)

# COMMAND ----------

best = results.sort_values("roc_auc", ascending=False).iloc[0]
BEST_RUN_ID = best["run_id"]

print("MEJOR MODELO")
print("Run:", best["run_name"])
print("Modelo:", best["modelo"])
print("ROC-AUC:", best["roc_auc"])
print("Run ID:", BEST_RUN_ID)

# COMMAND ----------

MODEL_URI = f"runs:/{BEST_RUN_ID}/model"

try:
    mv = mlflow.register_model(MODEL_URI, REGISTERED_MODEL)
    print("Modelo registrado:", mv.name)
    print("Versión:", mv.version)
except Exception as e:
    print("No se pudo registrar.")
    print("Si el error indica que falta model signature, vuelve a ejecutar el Notebook 03 agregando infer_signature al log_model.")
    raise

# COMMAND ----------

client.set_registered_model_alias(
    REGISTERED_MODEL,
    ALIAS,
    mv.version
)

principal = client.get_model_version_by_alias(REGISTERED_MODEL, ALIAS)

print("================================")
print("MODELO REGISTRADO")
print("Modelo:", principal.name)
print("Versión:", principal.version)
print("Alias:", ALIAS)
print("Run ID:", principal.run_id)
print("================================")

# MAGIC %md
# MAGIC **Evidencia:** Catalog Explorer → workspace → control2 → Models → `modelo_control2`.
# MAGIC Captura donde aparezcan versión y alias `PRINCIPAL`.
Control2